In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('../data/processed/model_ready_data.csv')

In [3]:
drop_cols_clf = ['Date', 'Revenue', 'ROI', 'Total_Cost', 'Profit_Flag']
X = df.drop(columns=drop_cols_clf)
y = df['Profit_Flag']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))

(133029, 30) (33258, 30)
Profit_Flag
Profit    0.770539
Loss      0.229461
Name: proportion, dtype: float64


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


In [7]:
y_train_enc = y_train.map({'Loss': 0, 'Profit': 1})
y_test_enc = y_test.map({'Loss': 0, 'Profit': 1})

In [8]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train_enc)
    preds = model.predict(X_test)
    
    acc = accuracy_score(y_test_enc, preds)
    prec = precision_score(y_test_enc, preds, pos_label=0)   # 0 = Loss now
    rec = recall_score(y_test_enc, preds, pos_label=0)
    f1 = f1_score(y_test_enc, preds, pos_label=0)
    
    print(f"\n{name}")
    print(f"Accuracy: {acc:.4f} | Precision(Loss): {prec:.4f} | Recall(Loss): {rec:.4f} | F1(Loss): {f1:.4f}")

/home/eni/dev-env/Marketing Campaign Performance Prediction/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Logistic Regression
Accuracy: 0.9158 | Precision(Loss): 0.8563 | Recall(Loss): 0.7610 | F1(Loss): 0.8058

Random Forest
Accuracy: 0.9173 | Precision(Loss): 0.8643 | Recall(Loss): 0.7589 | F1(Loss): 0.8082

XGBoost
Accuracy: 0.9176 | Precision(Loss): 0.8643 | Recall(Loss): 0.7605 | F1(Loss): 0.8091


In [9]:
import pickle

with open('../model/classification_model.pkl', 'wb') as f:
    pickle.dump(models['XGBoost'], f)

with open('../model/classification_features.pkl', 'wb') as f:
    pickle.dump(list(X_train.columns), f)

# save the label mapping too — app needs this to decode predictions back to 'Profit'/'Loss'
import json
with open('../model/label_mappings.json', 'w') as f:
    json.dump({'Loss': 0, 'Profit': 1}, f)